# 02 — Fine-tune XLM-RoBERTa (Colab GPU)

**Runtime → Change runtime type → GPU (T4).** Trains the primary bilingual transformer on the 3-class corpus, evaluates on the test split, and saves the model. Download `models/xlmr/` afterwards (or mount Drive) for the local app.

Optionally trains BanglaBERT for the Bangla-only comparison in the report.

In [ ]:
# --- Colab setup: clone/upload the project, then install deps. ---
# If you uploaded the repo as a folder, set PROJECT_DIR accordingly.
import os
PROJECT_DIR = '/content/Fake-News-Detection'   # adjust if needed
if not os.path.isdir(PROJECT_DIR):
    print('Upload the project to', PROJECT_DIR, 'or git clone it here.')
%cd $PROJECT_DIR
!pip -q install -r requirements.txt

In [ ]:
import sys, os
ROOT = os.path.abspath('.')
if ROOT not in sys.path: sys.path.insert(0, ROOT)
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Build corpus (or upload data/processed/*.parquet produced locally by notebook 01).
from src.data import build_corpus
build_corpus.build()

In [ ]:
# Fine-tune XLM-RoBERTa (primary). Saves to models/xlmr + reports/xlmr_metrics.json
from src.models.transformer import train_transformer
train_transformer(tag='xlmr')

In [ ]:
# OPTIONAL: BanglaBERT for Bangla-only comparison (report table).
from src.config import load_config
bn = load_config()['transformer']['bangla_model_name']
train_transformer(model_name=bn, tag='banglabert')

In [ ]:
# Build report assets (comparison table, confusion matrices, architecture diagram).
from src import report_assets
report_assets.main()

In [ ]:
# Zip the fine-tuned model for download to your local machine.
!zip -r -q models_xlmr.zip models/xlmr reports
from google.colab import files  # type: ignore
files.download('models_xlmr.zip')